# Neural-Network Evaluation for PUF Datasets

This notebook trains and evaluates state-of-the-art MLP models on generated PUF datasets.
It is intended to measure how well a neural network can learn the mapping from challenge features to binary PUF responses.

The workflow includes:
- loading a saved dataset,
- splitting it into training and test sets,
- training an MLP with configurable architecture and hyperparameters,
- reporting accuracy for different dataset sizes.

You can configure the dataset file to be tested and the network size as needed.

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np


# -----------------------------------------------------------------------------
# MLP training helper for PUF classification
# -----------------------------------------------------------------------------
def train_mlp_puf(X, y, net=[128, 32, 16], lr=0.001, epochs=20, bs=1000, seed=1, early_stop=None):
    """Train a small MLP on PUF challenge-response data."""

    # Fix random seeds for reproducibility.
    torch.manual_seed(seed)
    np.random.seed(seed)

    # Select the computation device: GPU if available, otherwise CPU.
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # Convert labels from {-1, 1} to {0, 1} for binary classification.
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    # Create a PyTorch dataset and data loader for mini-batch training.
    dataset = TensorDataset(X_t, y_t)
    loader = DataLoader(dataset, batch_size=bs, shuffle=True)

    # Build the MLP architecture from the provided hidden layer sizes.
    layers = []
    in_dim = X.shape[1]

    for h in net:
        layers.append(nn.Linear(in_dim, h))
        layers.append(nn.ReLU())
        in_dim = h

    layers.append(nn.Linear(in_dim, 1))
    layers.append(nn.Sigmoid())  # BCE loss expects probabilities in [0, 1]

    model = nn.Sequential(*layers).to(device)

    # Use Adam optimizer and binary cross-entropy loss.
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()

    # Training loop over epochs.
    for epoch in range(epochs):
        total_loss = 0

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            pred = model(xb)
            loss = loss_fn(pred, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item()

        # Evaluate accuracy after each epoch.
        acc = evaluate_mlp(model, X_t, y_t, device)

        print(f"Epoch {epoch+1}/{epochs} | Loss={total_loss:.4f} | Acc={acc:.4f}")

        if early_stop is not None and acc >= early_stop:
            print("Early stop triggered.")
            break

    return model


# -----------------------------------------------------------------------------
# Evaluation helper for the trained MLP
# -----------------------------------------------------------------------------
def evaluate_mlp(model, X_t, y_t, device, batch_size=2048):
    """Evaluate model accuracy on a batch-wise basis."""
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for i in range(0, len(X_t), batch_size):
            xb = torch.tensor(X_t[i:i+batch_size], dtype=torch.float32).to(device)
            yb = torch.tensor(y_t[i:i+batch_size], dtype=torch.float32).view(-1, 1)

            preds = model(xb).cpu()
            preds = (preds > 0.5).int()

            # Convert predictions and labels back to {-1, +1} for comparison.
            preds = 2 * preds - 1
            yb = 2 * yb.int() - 1

            correct += (preds == yb).sum().item()
            total += yb.size(0)

    return correct / total


In [ ]:
"""
Dataset evaluation workflow
--------------------------
This cell loads a saved PUF dataset, splits it into train/test sets, trains an
MLP, and reports the accuracy for several dataset sizes.
"""

import numpy as np
import torch
from sklearn.model_selection import train_test_split
import csv
import os

# -----------------------------------------------------------------------------
# Load the dataset to evaluate
# -----------------------------------------------------------------------------
dataset_file = "puf_data_novel_up1_down5_puf0.npz"  # File name of the PUF dataset to evaluate
 data = np.load(dataset_file)
net = [128, 32, 16]  # Example MLP architecture: [128, 32, 16]
X = data["X"]
y = data["y"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = []

# -----------------------------------------------------------------------------
# Define the dataset sizes to evaluate
# -----------------------------------------------------------------------------
# Use powers of ten and a few intermediate values for a simple scaling study.
base_powers = [10**p for p in range(3, 8)]  # 1e3 to 1e7

for n in base_powers:
    if n > len(X):
        print(f"Skipping {n} (only {len(X)} available)")
        continue

    print(f"\nDataset size: {n}")

    # Take the first n samples from the dataset.
    X_n = X[:int(n)]
    y_n = y[:int(n)]

    # Split the selected samples into training and test sets.
    X_train, X_test, y_train, y_test = train_test_split(
        X_n, y_n, test_size=0.2, random_state=23
    )

    # Train the MLP on the training split and evaluate on the test split.
    model = train_mlp_puf(X_train, y_train, net=net)
    model.to(device)

    acc = evaluate_mlp(model, X_test, y_test, device)
    print(f"Accuracy @ {n}: {acc:.4f}")

    results.append((n, acc))

# -----------------------------------------------------------------------------
# Save the accuracies to a CSV file
# -----------------------------------------------------------------------------
base_name = os.path.splitext(os.path.basename(dataset_file))[0]
output_file = f"{base_name}_accuracy.csv"

with open(output_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["dataset_size", "accuracy"])
    writer.writerows(results)

print(f"\nSaved accuracies to {output_file}")



Dataset size: 1000
Using device: cuda
Epoch 1/20 | Loss=0.6936 | Acc=0.5050
Epoch 2/20 | Loss=0.6915 | Acc=0.5787
Epoch 3/20 | Loss=0.6896 | Acc=0.6288
Epoch 4/20 | Loss=0.6876 | Acc=0.6725
Epoch 5/20 | Loss=0.6857 | Acc=0.7050
Epoch 6/20 | Loss=0.6836 | Acc=0.7163
Epoch 7/20 | Loss=0.6814 | Acc=0.7362
Epoch 8/20 | Loss=0.6791 | Acc=0.7475
Epoch 9/20 | Loss=0.6767 | Acc=0.7575
Epoch 10/20 | Loss=0.6740 | Acc=0.7588
Epoch 11/20 | Loss=0.6712 | Acc=0.7688
Epoch 12/20 | Loss=0.6681 | Acc=0.7738
Epoch 13/20 | Loss=0.6648 | Acc=0.7775
Epoch 14/20 | Loss=0.6612 | Acc=0.7788
Epoch 15/20 | Loss=0.6573 | Acc=0.7825
Epoch 16/20 | Loss=0.6530 | Acc=0.7887
Epoch 17/20 | Loss=0.6483 | Acc=0.7925
Epoch 18/20 | Loss=0.6432 | Acc=0.7937
Epoch 19/20 | Loss=0.6377 | Acc=0.7975
Epoch 20/20 | Loss=0.6317 | Acc=0.8025
Accuracy @ 1000: 0.5200

Saved accuracies to puf_data_novel_up1_down5_puf0_accuracy.csv


/tmp/ipykernel_3883178/123374330.py:84: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  xb = torch.tensor(X_t[i:i+batch_size],dtype=torch.float32).to(device)
/tmp/ipykernel_3883178/123374330.py:85: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  yb = torch.tensor(y_t[i:i+batch_size],dtype=torch.float32).view(-1, 1)
